# Best Practices — Tests

Unit test for `compute_user_region_stats`. The test builds tiny in-memory DataFrames as input, calls the function from the Functions notebook, and asserts against a hand-built expected DataFrame using `pyspark.testing.assertDataFrameEqual` (available since PySpark 3.5).

The fixture covers two users:
* user 1 in France with five answers (scores 10, 20, 5, 8, 12) — survives the `min_answers >= 5` filter; expected `region=europe`, `answer_count=5`, `avg_score=11.0`
* user 2 in USA with one answer — filtered out by the threshold

In a real codebase this would be a `tests/test_transformations.py` file run by `pytest` in CI, with a session-scoped `SparkSession` fixture in `conftest.py`. The notebook version uses `%run` to share the same function definitions the pipeline uses.

In [1]:
from pyspark.sql import SparkSession
from pyspark.testing import assertDataFrameEqual

In [2]:
spark = (
    SparkSession
    .builder
    .appName('Best Practices - Tests')
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/16 15:24:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/16 15:24:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/16 15:24:44 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


#### Include the Functions notebook

Same `%run` mechanism as the pipeline notebook — this guarantees the tests assert against exactly the same `compute_user_region_stats` definition that the pipeline uses.

In [3]:
%run "./Best Practices - Functions.ipynb"

#### Fixtures

In [4]:
users_test = spark.createDataFrame(
    [
        (1, 'France'),
        (2, 'USA'),
    ],
    ['user_id', 'location'],
)

answers_test = spark.createDataFrame(
    [
        (101, 1, 10),
        (102, 1, 20),
        (103, 1,  5),
        (104, 1,  8),
        (105, 1, 12),  # user 1: five answers — survives the >= 5 filter
        (106, 2,  7),  # user 2: one answer  — filtered out
    ],
    ['answer_id', 'user_id', 'score'],
)

expected = spark.createDataFrame(
    [
        (1, 'europe', 5, 11.0),
    ],
    ['user_id', 'region', 'answer_count', 'avg_score'],
)

#### Run and assert

`assertDataFrameEqual` checks schema + values (with a tolerance for floats) and ignores row order by default.

In [5]:
actual = compute_user_region_stats(users_test, answers_test, min_answers=5)

assertDataFrameEqual(actual, expected)

print('OK')

/Users/davidvrba/Library/Caches/pypoetry/virtualenvs/apache-spark-advanced-topics-DKWxjOIK-py3.11/lib/python3.11/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(
[Stage 1:=============================>                             (4 + 4) / 8]

OK


In [ ]:
spark.stop()